# Retail medallion silver transform

This notebook applies validation, deduplication, anomaly quarantine, customer and product enrichment,
and incremental-style merge logic from bronze into conformed silver tables.


In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = "2026-03-02"
PIPELINE_RUN_ID = "local-demo"
environment = ENVIRONMENT
processing_date = PROCESSING_DATE
pipeline_run_id = PIPELINE_RUN_ID


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window
import json

purchases_raw_table = 'bronze.retail_purchases_raw'
customers_raw_table = 'bronze.retail_customers_raw'
products_raw_table = 'bronze.retail_products_raw'

customers_silver_table = 'silver.retail_customers'
products_silver_table = 'silver.retail_products'
sales_silver_table = 'silver.retail_sales_enriched'
quarantine_table = 'silver.retail_purchases_quarantine'
audit_table = 'silver.retail_processing_audit'


In [ ]:
def log_event(stage, message, **metrics):
    record = {
        'stage': stage,
        'message': message,
        'environment': environment,
        'processing_date': processing_date,
        'pipeline_run_id': pipeline_run_id,
        'metrics': metrics,
    }
    print(json.dumps(record, sort_keys=True))

cutoff_timestamp = F.to_timestamp(F.lit(f'{processing_date}T23:59:59Z'))

purchase_window = Window.partitionBy('purchase_id').orderBy(
    F.col('ingest_ts').desc_nulls_last(),
    F.col('metadata_ingestion_timestamp').desc_nulls_last(),
)
customer_window = Window.partitionBy('customer_id').orderBy(F.col('metadata_ingestion_timestamp').desc_nulls_last())
product_window = Window.partitionBy('product_id').orderBy(F.col('metadata_ingestion_timestamp').desc_nulls_last())

purchases_df = spark.table(purchases_raw_table).filter(F.col('ingest_ts') <= cutoff_timestamp)
customers_df = spark.table(customers_raw_table)
products_df = spark.table(products_raw_table)


In [ ]:
conformed_customers_df = (
    customers_df
    .withColumn('customer_rank', F.row_number().over(customer_window))
    .filter(F.col('customer_rank') == 1)
    .drop('customer_rank')
    .withColumn('customer_key', F.xxhash64('customer_id'))
    .select(
        'customer_key', 'customer_id', 'customer_name', 'loyalty_tier',
        'home_region', 'signup_date', 'email_domain', 'synthetic_record'
    )
)

conformed_products_df = (
    products_df
    .withColumn('product_rank', F.row_number().over(product_window))
    .filter(F.col('product_rank') == 1)
    .drop('product_rank')
    .withColumn('product_key', F.xxhash64('product_id'))
    .withColumn('is_active', F.when(F.lower(F.col('is_active')) == 'true', F.lit(True)).otherwise(F.lit(False)))
    .select(
        'product_key', 'product_id', 'product_name', 'category', 'subcategory',
        'unit_price', 'unit_cost', 'is_active', 'synthetic_record'
    )
)


In [ ]:
ranked_purchases_df = purchases_df.withColumn('purchase_rank', F.row_number().over(purchase_window))

duplicate_quarantine_df = (
    ranked_purchases_df
    .filter(F.col('purchase_rank') > 1)
    .withColumn('quarantine_reason', F.lit('duplicate_purchase'))
    .withColumn('quarantine_stage', F.lit('deduplication'))
    .withColumn('quarantine_timestamp', F.current_timestamp())
)

latest_purchase_df = ranked_purchases_df.filter(F.col('purchase_rank') == 1).drop('purchase_rank')

anomaly_condition = (
    (F.col('anomaly_label') != F.lit('none'))
    | F.col('customer_id').isNull()
    | F.col('product_id').isNull()
    | F.col('quantity').isNull()
    | (F.col('quantity') <= 0)
    | F.col('unit_price').isNull()
    | (F.col('unit_price') < 0)
)

anomaly_quarantine_df = (
    latest_purchase_df
    .filter(anomaly_condition)
    .withColumn(
        'quarantine_reason',
        F.concat_ws(
            '; ',
            F.when(F.col('anomaly_label') != 'none', F.col('anomaly_label')),
            F.when(F.col('customer_id').isNull(), F.lit('null_customer_id')),
            F.when(F.col('product_id').isNull(), F.lit('null_product_id')),
            F.when(F.col('quantity').isNull() | (F.col('quantity') <= 0), F.lit('invalid_quantity')),
            F.when(F.col('unit_price').isNull() | (F.col('unit_price') < 0), F.lit('invalid_unit_price')),
        ),
    )
    .withColumn('quarantine_stage', F.lit('quality_rules'))
    .withColumn('quarantine_timestamp', F.current_timestamp())
)

valid_purchase_df = latest_purchase_df.filter(~anomaly_condition)


In [ ]:
enriched_purchase_df = (
    valid_purchase_df.alias('p')
    .join(conformed_customers_df.alias('c'), on='customer_id', how='left')
    .join(conformed_products_df.alias('pr'), on='product_id', how='left')
    .withColumn('purchase_date', F.to_date('purchase_ts'))
    .withColumn('year_month', F.date_format('purchase_ts', 'yyyy-MM'))
    .withColumn('sales_key', F.xxhash64('purchase_id'))
    .withColumn('effective_unit_cost', F.coalesce(F.col('pr.unit_cost'), F.round(F.col('p.unit_price') * F.lit(0.62), 2)))
    .withColumn('revenue', F.round(F.col('p.quantity') * F.col('p.unit_price'), 2))
    .withColumn('cost', F.round(F.col('p.quantity') * F.col('effective_unit_cost'), 2))
    .withColumn('gross_margin', F.round(F.col('revenue') - F.col('cost'), 2))
    .withColumn('gross_margin_pct', F.round(F.when(F.col('revenue') != 0, F.col('gross_margin') / F.col('revenue')), 6))
)

referential_quarantine_df = (
    enriched_purchase_df
    .filter(F.col('customer_key').isNull() | F.col('product_key').isNull())
    .withColumn(
        'quarantine_reason',
        F.concat_ws(
            '; ',
            F.when(F.col('customer_key').isNull(), F.lit('missing_customer_dimension')),
            F.when(F.col('product_key').isNull(), F.lit('missing_product_dimension')),
        ),
    )
    .withColumn('quarantine_stage', F.lit('dimension_join'))
    .withColumn('quarantine_timestamp', F.current_timestamp())
)

valid_sales_df = (
    enriched_purchase_df
    .filter(F.col('customer_key').isNotNull() & F.col('product_key').isNotNull())
    .select(
        'sales_key', 'purchase_id', 'purchase_ts', 'ingest_ts', 'purchase_date', 'year_month',
        'customer_key', 'customer_id', 'customer_name', 'loyalty_tier', 'home_region', 'signup_date', 'email_domain',
        'product_key', 'product_id', 'product_name', 'category', 'subcategory',
        F.col('p.unit_price').alias('unit_price'),
        F.col('effective_unit_cost').alias('unit_cost'),
        F.col('pr.is_active').alias('is_active'),
        'store_id', 'channel', 'quantity', 'revenue', 'cost', 'gross_margin', 'gross_margin_pct',
        'synthetic_record', 'anomaly_label'
    )
)

quarantine_df = duplicate_quarantine_df.unionByName(anomaly_quarantine_df, allowMissingColumns=True)         .unionByName(referential_quarantine_df, allowMissingColumns=True)


In [ ]:
conformed_customers_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(customers_silver_table)
conformed_products_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(products_silver_table)
quarantine_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(quarantine_table)

if spark.catalog.tableExists(sales_silver_table):
    target = DeltaTable.forName(spark, sales_silver_table)
    (
        target.alias('target')
        .merge(valid_sales_df.alias('source'), 'target.purchase_id = source.purchase_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        valid_sales_df.write.format('delta')
        .mode('overwrite')
        .partitionBy('year_month')
        .option('overwriteSchema', 'true')
        .saveAsTable(sales_silver_table)
    )

spark.sql(f'OPTIMIZE {sales_silver_table} ZORDER BY (purchase_date, customer_id, product_id)')

audit_df = spark.createDataFrame([
    (processing_date, pipeline_run_id, valid_sales_df.count(), quarantine_df.count())
], 'processing_date string, pipeline_run_id string, valid_sales_rows long, quarantine_rows long').withColumn('logged_at', F.current_timestamp())
audit_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(audit_table)

log_event(
    'silver_validation',
    'Retail silver transformation complete',
    customer_rows=conformed_customers_df.count(),
    product_rows=conformed_products_df.count(),
    valid_sales_rows=valid_sales_df.count(),
    quarantine_rows=quarantine_df.count(),
)
spark.sql(f'SELECT COUNT(*) AS silver_sales_rows FROM {sales_silver_table}').show()
spark.sql(f'SELECT COUNT(*) AS silver_quarantine_rows FROM {quarantine_table}').show()
